# Housing Price Prediction using FPGA Matrix-Vector Multiplication

**Files required in this directory on PYNQ:**
- `mat_vec_stream1.bit` — bitstream
- `mat_vec_stream1.hwh` — rename `design_1.hwh` from hw_files
- `housing.csv` — dataset (5000 samples, 5 features + price target)

Uses the streaming HLS kernel from Problem 1 (`m_v_st_0`, SIZE=10000) to accelerate
linear regression on the housing dataset. The normal equation is solved with FPGA
matrix-vector products replacing `np.dot` for vector outputs.

Data is z-score standardised before conversion to int32 so all values fit safely
within int32 when accumulated over 5000 samples (max ~±4.5 × 10⁸ vs int32 limit ~2.1 × 10⁹).

In [ ]:
import numpy as np
from pynq import Overlay, allocate
import pynq.lib.dma

SIZE  = 10000  # fixed in hardware
SCALE = 100    # float * SCALE -> int32  (safe after z-score standardisation)

print("Programming the FPGA...")
ol = Overlay('./mat_vec_stream1.bit')

# Start HLS kernel once with auto-restart so it accepts repeated DMA bursts
ol.m_v_st_0.write(0x0, 0x81)  # ap_start=1, auto_restart=1

dma0 = ol.axi_dma_0  # sendchannel -> in_M,  recvchannel <- out
dma1 = ol.axi_dma_1  # sendchannel -> in_V
print("FPGA ready.")

In [ ]:
# Load housing.csv — columns: Income, Age, Rooms, Bedrooms, Population, Price, Address
data  = np.genfromtxt('housing.csv', delimiter=',', skip_header=1, usecols=range(6))
X_raw = data[:, :5]   # (5000, 5)
y_raw = data[:, 5]    # (5000,)  — Price in dollars

# Z-score standardise so values sit in ~[-3, 3] before SCALE multiplication
X_mean, X_std = X_raw.mean(axis=0), X_raw.std(axis=0)
y_mean, y_std = y_raw.mean(),        y_raw.std()

X_norm = (X_raw - X_mean) / X_std
y_norm = (y_raw - y_mean) / y_std

# Augment with bias column
X = np.hstack([np.ones((X_norm.shape[0], 1)), X_norm])  # (5000, 6)

print(f"Samples: {X.shape[0]},  Features (including bias): {X.shape[1]}")
print(f"Price — min: ${y_raw.min():,.0f}, max: ${y_raw.max():,.0f}, mean: ${y_raw.mean():,.0f}")

In [ ]:
def hw_matvec(A_int, v_int):
    """Compute A_int @ v_int using the FPGA streaming kernel.

    A_int : (rows, cols) int32 array — padded internally to SIZE x SIZE
    v_int : (cols,)      int32 array — padded internally to SIZE
    Returns (rows,) int32 result.
    """
    rows, cols = A_int.shape

    M_buf   = allocate(shape=(SIZE, SIZE), dtype=np.int32)
    V_buf   = allocate(shape=(SIZE,),      dtype=np.int32)
    Out_buf = allocate(shape=(SIZE,),      dtype=np.int32)

    M_buf[:rows, :cols] = A_int
    V_buf[:cols]        = v_int

    # Match problem-1 transfer order: recv first, then V, then M
    dma0.recvchannel.transfer(Out_buf)
    dma1.sendchannel.transfer(V_buf)
    dma0.sendchannel.transfer(M_buf)

    dma0.sendchannel.wait()
    dma1.sendchannel.wait()
    dma0.recvchannel.wait()

    result = np.array(Out_buf[:rows], dtype=np.int32)
    del M_buf, V_buf, Out_buf
    return result

In [ ]:
def train_linear_regression_hw(X, y):
    """Solve the normal equation using FPGA for matrix-vector products.

    beta = inv(X^T X) @ X^T y
    Both X and y are scaled by SCALE before the FPGA call; the factors
    cancel so beta is returned in the original (standardised) float units.
    """
    X_int = (X * SCALE).astype(np.int32)
    y_int = (y * SCALE).astype(np.int32)

    XT       = X_int.T                                             # (6, 5000)
    XT_X     = XT.astype(np.float64) @ X_int.astype(np.float64)   # (6, 6)  numpy — matrix×matrix
    XT_X_inv = np.linalg.inv(XT_X)                                 # (6, 6)

    XT_y_int = hw_matvec(XT, y_int)                                # (6,)    FPGA
    XT_y     = XT_y_int.astype(np.float64)

    # inv(S^2 * A) @ (S^2 * b)  ==  inv(A) @ b  — scale cancels exactly
    return XT_X_inv @ XT_y


def predict_hw(X, beta):
    """Predict y = X @ beta using the FPGA kernel. Returns standardised output."""
    X_int    = (X    * SCALE).astype(np.int32)
    beta_int = (beta * SCALE).astype(np.int32)

    y_hat_int = hw_matvec(X_int, beta_int)          # (5000,)  FPGA
    return y_hat_int.astype(np.float64) / (SCALE * SCALE)

In [ ]:
print("Training (FPGA computes X^T y)...")
beta = train_linear_regression_hw(X, y_norm)

print("Predicting (FPGA computes X @ beta)...")
y_pred_norm = predict_hw(X, beta)
y_pred = y_pred_norm * y_std + y_mean   # un-standardise back to dollars


def print_results(y_true, y_pred, beta):
    feature_names = ['bias', 'Income', 'House Age', 'Rooms', 'Bedrooms', 'Population']

    print("\n=== Model Coefficients (standardised space) ===")
    for name, coef in zip(feature_names, beta):
        print(f"  {name:<12}: {coef:10.4f}")

    mse  = np.mean((y_true - y_pred) ** 2)
    rmse = np.sqrt(mse)
    r2   = 1 - np.sum((y_true - y_pred) ** 2) / np.sum((y_true - np.mean(y_true)) ** 2)

    print("\n=== Predictions vs Actual (first 10 samples) ===")
    print(f"{'#':>4}  {'Predicted ($)':>14}  {'Actual ($)':>14}  {'Error ($)':>12}")
    for i in range(10):
        print(f"{i:>4}  {y_pred[i]:>14,.0f}  {y_true[i]:>14,.0f}  {y_pred[i]-y_true[i]:>12,.0f}")

    print(f"\n  MSE : ${mse:>14,.2f}")
    print(f"  RMSE: ${rmse:>14,.2f}")
    print(f"  R²  :  {r2:.4f}")


print_results(y_raw, y_pred, beta)

In [ ]:
del ol
print("Finished!")